In [1]:
import pandas as pd
import numpy as np
from scipy.special import eval_legendre
from sklearn.preprocessing import MinMaxScaler

df_raw = pd.read_csv('raw_orbit_data.csv')
df_proc = df_raw.copy()

mu = 3.986004418e14  # Standart Earth gravitational parameter (m^3/s^2)
Re = 6378137.0       # Earth ekvatorial radius (m)

max_degree = 6
lifted_features = {}

lifted_features['mean_motion_n'] = np.sqrt(mu / (df_proc['a']**3))

p = df_proc['a'] * (1 - df_proc['e']**2)
J2_factor = (Re / p)**2

lifted_features['J2_raan_term'] = J2_factor * np.cos(df_proc['i'])
lifted_features['J2_omega_term'] = J2_factor * (4 - 5 * (np.sin(df_proc['i'])**2))

scaler_ae = MinMaxScaler(feature_range=(-1, 1))
df_proc[['a', 'e']] = scaler_ae.fit_transform(df_proc[['a', 'e']])

df_phys = pd.DataFrame(lifted_features)
scaler_phys = MinMaxScaler(feature_range=(-1, 1))
df_phys[df_phys.columns] = scaler_phys.fit_transform(df_phys)

math_features = {}

for col in ['a', 'e']:
    x = df_proc[col].values
    for degree in range(2, max_degree + 1):
        math_features[f"{col}_P{degree}"] = eval_legendre(degree, x)

# Fourier
for angle_col in ['i', 'omega', 'raan', 'm']:
    angle = df_proc[angle_col].values
    for k in range(1, max_degree + 1):
        math_features[f"sin_{k}{angle_col}"] = np.sin(k * angle)
        math_features[f"cos_{k}{angle_col}"] = np.cos(k * angle)

df_math = pd.DataFrame(math_features)
df_final = pd.concat([df_proc, df_phys, df_math], axis=1)

columns_to_remove = [
    'cos_3m', # Removing this improves error by 16.95%
    'a_P6', # Removing this improves error by 4.56%
    'cos_6raan', # Removing this improves error by 3.59%
    'cos_5raan', # Removing this improves error by 3.89%
]

# Eğer listede geçerli kolonlar varsa, onları sistemden kalıcı olarak at:
kolonlari_sil = [col for col in columns_to_remove if col in df_final.columns]
if kolonlari_sil:
    df_final = df_final.drop(columns=kolonlari_sil)

# Zaman sütunları yanlışlıkla girdiyse güvenlik amaçlı sil
guvenlik_sil = [c for c in ['t', 't_hours'] if c in df_final.columns]
if guvenlik_sil:
    df_final = df_final.drop(columns=guvenlik_sil)

# Adds an overall improvement of 49.16%
df_final['(J2_raan_term * cos_4omega)'] = df_final['J2_raan_term'] * df_final['cos_4omega']

# Adds an overall improvement of 24.11%
df_final['(cos_4omega * cos_2m)'] = df_final['cos_4omega'] * df_final['cos_2m']

# Adds an overall improvement of 19.34%
df_final['(sin_6i * cos_1omega)'] = df_final['sin_6i'] * df_final['cos_1omega']

# Adds an overall improvement of 4.97%
df_final['(e_P3 * (J2_raan_term * cos_4omega))'] = df_final['e_P3'] * df_final['(J2_raan_term * cos_4omega)']

# Adds an overall improvement of 6.21%
df_final['(m * sin_5m)'] = df_final['m'] * df_final['sin_5m']

df_final.to_csv('lifted_orbit_data.csv', index=False)

print("done")

done


In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import MinMaxScaler
from IPython.display import display
import time

# ==============================================================================
# 1. DYNAMIC DATA LOADING & PREPARATION
# ==============================================================================
print("Loading data...")
try:
    df_raw = pd.read_csv("raw_orbit_data.csv")
    df_lifted = pd.read_csv("lifted_orbit_data.csv")
except FileNotFoundError as e:
    raise FileNotFoundError(f"Missing file: {e}. Ensure both 'raw_orbit_data.csv' and 'lifted_orbit_data.csv' are in the directory.")

# Remove time columns
raw_state_cols = [c for c in df_raw.columns if c not in ['t', 't_hours']]
lifted_state_cols = [c for c in df_lifted.columns if c not in ['t', 't_hours']]

df_raw = df_raw[raw_state_cols]
df_lifted = df_lifted[lifted_state_cols]

n_samples = len(df_lifted)
print(f"Total Samples: {n_samples}")

# Prepare Scaler exactly as in the original code
scaler_ae = MinMaxScaler(feature_range=(-1, 1))
scaler_ae.fit(df_raw[['a', 'e']])

# Core columns absolutely required for unlifting (Inverse Transform & Arctan2)
CORE_COLS = ['a', 'e', 'i', 'sin_1omega', 'cos_1omega', 'sin_1raan', 'cos_1raan', 'sin_1m', 'cos_1m']
for c in CORE_COLS:
    if c not in df_lifted.columns:
        raise ValueError(f"Critical column '{c}' not found in lifted_orbit_data.csv!")

# ==============================================================================
# 2. DYNAMIC TIME WINDOWS (3-FOLD EXPANDING CROSS VALIDATION)
# ==============================================================================
p60 = int(n_samples * 0.6)
p70 = int(n_samples * 0.7)
p80 = int(n_samples * 0.8)
p90 = int(n_samples * 0.9)

# Expanding training windows (Train always starts from 0)
cv_splits = [
    {'train_end': p60, 'test_end': p70},
    {'train_end': p70, 'test_end': p80},
    {'train_end': p80, 'test_end': p90}
]

# ==============================================================================
# 3. REAL PHYSICAL SPACE (UNLIFTED) EVALUATION ENGINE
# ==============================================================================
def angle_error(true_vals, pred_vals):
    """Mean Absolute Error (MAE) for angles in radians."""
    return np.mean(np.abs(np.arctan2(np.sin(true_vals - pred_vals), np.cos(true_vals - pred_vals))))

def evaluate_feature_set(selected_features):
    """
    Trains the model using specified features, unlifts the predictions, 
    and calculates MAE in the raw (real) physical space.
    """
    # Reject if any critical column is missing
    if not all(c in selected_features for c in CORE_COLS):
        return np.inf, 1000.0, {}

    Z_data = df_lifted[selected_features].values
    
    cv_overall_errors = []
    feature_errors_dict = {col: [] for col in ['a', 'e', 'i', 'omega', 'raan', 'm']}
    max_eig_overall = 0
    
    for split in cv_splits:
        Z_train = Z_data[:split['train_end']]
        
        # FUTURE TRUE VALUES (Taken directly from raw_orbit_data)
        df_raw_future = df_raw.iloc[split['train_end']:split['test_end']].reset_index(drop=True)
        N_forecast_steps = len(df_raw_future)
        
        Z1, Z2 = Z_train[:-1], Z_train[1:]
        z_current = Z_train[-1].copy()
        
        # 1. Model Training
        model = Ridge(alpha=1e-7, fit_intercept=False)
        model.fit(Z1, Z2)
        K = model.coef_
        
        # Stability Check
        max_eig = np.max(np.abs(np.linalg.eigvals(K)))
        max_eig_overall = max(max_eig_overall, max_eig)
        if max_eig > 1.01:
            return np.inf, max_eig, {} # Unstable
            
        # 2. Recursive Forecast
        Z_forecast = []
        for _ in range(N_forecast_steps):
            z_next = z_current @ K.T
            Z_forecast.append(z_next)
            z_current = z_next.copy()
            
        Z_forecast = np.array(Z_forecast)
        
        # 3. Unlift (Inverse Transform)
        pred_df = pd.DataFrame()
        
        # Convert a and e back to real scale
        a_idx = selected_features.index('a')
        e_idx = selected_features.index('e')
        pred_df[['a', 'e']] = scaler_ae.inverse_transform(Z_forecast[:, [a_idx, e_idx]])
        
        # Inclination remains direct
        pred_df['i'] = Z_forecast[:, selected_features.index('i')]
        
        # Reconstruct angles using Arctan2
        pred_df['omega'] = np.arctan2(Z_forecast[:, selected_features.index('sin_1omega')], Z_forecast[:, selected_features.index('cos_1omega')])
        pred_df['raan']  = np.arctan2(Z_forecast[:, selected_features.index('sin_1raan')],  Z_forecast[:, selected_features.index('cos_1raan')])
        pred_df['m']     = np.arctan2(Z_forecast[:, selected_features.index('sin_1m')],     Z_forecast[:, selected_features.index('cos_1m')])
        
        # 4. Calculate Mean Absolute Error (MAE)
        err_a = np.mean(np.abs(df_raw_future['a'].values - pred_df['a'].values))
        err_e = np.mean(np.abs(df_raw_future['e'].values - pred_df['e'].values))
        err_i = np.mean(np.abs(df_raw_future['i'].values - pred_df['i'].values))
        
        err_omega = angle_error(df_raw_future['omega'].values, pred_df['omega'].values)
        err_raan = angle_error(df_raw_future['raan'].values, pred_df['raan'].values)
        err_m = angle_error(df_raw_future['m'].values, pred_df['m'].values)
        
        feature_errors_dict['a'].append(err_a)
        feature_errors_dict['e'].append(err_e)
        feature_errors_dict['i'].append(err_i)
        feature_errors_dict['omega'].append(err_omega)
        feature_errors_dict['raan'].append(err_raan)
        feature_errors_dict['m'].append(err_m)
        
    # Average errors across all 3 CV windows
    final_errors = {col: np.mean(val) for col, val in feature_errors_dict.items()}
    return final_errors, max_eig_overall

# ==============================================================================
# 4. BASELINE MODEL & ABLATION LOOP
# ==============================================================================
print("\n[Step 1] Calculating Baseline Model (All Features)...")
current_features = list(df_lifted.columns)
base_errors, base_eig = evaluate_feature_set(current_features)

print(f"Baseline Max Eigenvalue: {base_eig:.4f}")
print("Baseline Errors (Physical Dimensions):")
for k, v in base_errors.items():
    print(f"  {k}: {v:.6e}")

print("\n[Step 2] Starting Feature Ablation Test (Removing one by one)...")
results = []
start_time = time.time()

# Features that can be removed (excluding CORE_COLS)
removable_features = [f for f in current_features if f not in CORE_COLS]

for i, feature in enumerate(removable_features):
    test_features = [f for f in current_features if f != feature]
    
    test_errors, max_eig = evaluate_feature_set(test_features)
    
    if max_eig <= 1.01 and isinstance(test_errors, dict):
        total_pct_improvement = 0
        row_data = {
            'Removed Feature': feature,
            'Max Eigenvalue': max_eig
        }
        
        # % Improvement for each physical target (Negative = Improved)
        for target in ['a', 'e', 'i', 'omega', 'raan', 'm']:
            b_err = base_errors[target]
            t_err = test_errors[target]
            
            pct_change = ((t_err - b_err) / (b_err + 1e-12)) * 100
            row_data[f'{target} Change (%)'] = pct_change
            total_pct_improvement += pct_change
            
        row_data['Overall Impact (%)'] = total_pct_improvement / 6.0
        results.append(row_data)

    if (i + 1) % 10 == 0:
        print(f"Tested feature: {i+1}/{len(removable_features)}")

# ==============================================================================
# 5. RESULTS TABLE & AUTO-GENERATED CODE
# ==============================================================================
df_results = pd.DataFrame(results)

if not df_results.empty:
    df_results = df_results.sort_values(by='Overall Impact (%)', ascending=True).reset_index(drop=True)
    
    print("\n" + "="*80)
    print("✅ MOST UNNECESSARY FEATURES (IMPROVES FORECAST WHEN REMOVED) ✅")
    print("Note: Negative (-) percentages indicate a REDUCTION in error (i.e., Model Improved).")
    print("="*80)
    
    def color_cells(val):
        if pd.isna(val) or not isinstance(val, (int, float)):
            return ''
        if val < -0.5:
            return 'color: #155724; background-color: #d4edda' # Improvement (Green)
        elif val > 0.5:
            return 'color: #721c24; background-color: #f8d7da' # Degradation (Red)
        return 'color: black'
        
    format_cols = [c for c in df_results.columns if '(%)' in c]
    format_dict = {col: "{:+.2f}%" for col in format_cols}
    format_dict['Max Eigenvalue'] = "{:.4f}"
    
    display(df_results.head(100).style.map(color_cells, subset=format_cols).format(format_dict))

    # CODE GENERATOR
    print("\n📋 AUTOMATIC REMOVAL LIST (Top 10 Worst Features) 📋")
    best_to_remove = df_results[df_results['Overall Impact (%)'] < -0.5].head(10)
    
    if not best_to_remove.empty:
        print("columns_to_remove = [")
        for idx, row in best_to_remove.iterrows():
            print(f"    '{row['Removed Feature']}', # Removing this improves error by {abs(row['Overall Impact (%)']):.2f}%")
        print("]")
    else:
        print("# No feature found that provides significant physical improvement when removed.")

Loading data...
Total Samples: 1862

[Step 1] Calculating Baseline Model (All Features)...
Baseline Max Eigenvalue: 1.0003
Baseline Errors (Physical Dimensions):
  a: 3.988509e+01
  e: 1.360213e-05
  i: 6.533699e-06
  omega: 9.480808e-03
  raan: 6.257420e-06
  m: 1.566333e-02

[Step 2] Starting Feature Ablation Test (Removing one by one)...
Tested feature: 10/59
Tested feature: 20/59
Tested feature: 30/59
Tested feature: 40/59
Tested feature: 50/59

✅ MOST UNNECESSARY FEATURES (IMPROVES FORECAST WHEN REMOVED) ✅
Note: Negative (-) percentages indicate a REDUCTION in error (i.e., Model Improved).


/home/cihat/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


,Removed Feature,Max Eigenvalue,a Change (%),e Change (%),i Change (%),omega Change (%),raan Change (%),m Change (%),Overall Impact (%)
0,sin_2omega,1.0003,+1.84%,+0.92%,-11.54%,+0.05%,+0.23%,-0.15%,-1.44%
1,sin_2i,1.0003,+0.04%,+0.03%,-3.59%,+0.00%,-0.03%,-0.00%,-0.59%
2,cos_3i,1.0003,+0.03%,+0.03%,-3.25%,+0.00%,-0.02%,-0.00%,-0.53%
3,sin_5i,1.0003,+0.04%,+0.03%,-3.15%,+0.00%,-0.02%,-0.00%,-0.52%
4,sin_1i,1.0003,+0.02%,+0.02%,-2.53%,+0.00%,-0.02%,-0.00%,-0.42%
5,cos_4i,1.0003,+0.03%,+0.02%,-2.39%,+0.00%,-0.01%,-0.00%,-0.39%
6,cos_4raan,1.0003,-0.01%,+0.03%,-0.65%,-0.01%,-1.30%,+0.01%,-0.32%
7,cos_6i,1.0003,+0.02%,+0.02%,-1.66%,+0.00%,-0.01%,-0.00%,-0.27%
8,J2_raan_term,1.0002,+0.06%,-0.03%,-1.15%,+0.13%,-0.35%,-0.19%,-0.26%
9,cos_1i,1.0003,+0.01%,+0.01%,-1.41%,+0.00%,-0.01%,-0.00%,-0.23%



📋 AUTOMATIC REMOVAL LIST (Top 10 Worst Features) 📋
columns_to_remove = [
    'sin_2omega', # Removing this improves error by 1.44%
    'sin_2i', # Removing this improves error by 0.59%
    'cos_3i', # Removing this improves error by 0.53%
    'sin_5i', # Removing this improves error by 0.52%
]


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import MinMaxScaler
from itertools import combinations
from IPython.display import display
import time

# ==============================================================================
# 1. DYNAMIC DATA LOADING & PREPARATION
# ==============================================================================
print("Loading data and initializing the Autonomous Pipeline...")
try:
    df_raw = pd.read_csv("raw_orbit_data.csv")
    df_lifted = pd.read_csv("lifted_orbit_data.csv")
except FileNotFoundError as e:
    raise FileNotFoundError(f"Missing file: {e}. Ensure both 'raw_orbit_data.csv' and 'lifted_orbit_data.csv' are in the directory.")

# Remove time columns
raw_state_cols = [c for c in df_raw.columns if c not in ['t', 't_hours']]
lifted_state_cols = [c for c in df_lifted.columns if c not in ['t', 't_hours']]

df_raw = df_raw[raw_state_cols]
df_lifted = df_lifted[lifted_state_cols]

feature_names = df_lifted.columns.tolist()
Z = df_lifted.values
n_samples, n_features = Z.shape
print(f"Total Samples: {n_samples} | Base Features: {n_features}")

# Prepare Scaler exactly as in the original code
scaler_ae = MinMaxScaler(feature_range=(-1, 1))
scaler_ae.fit(df_raw[['a', 'e']])

# Core columns absolutely required for unlifting
CORE_COLS = ['a', 'e', 'i', 'sin_1omega', 'cos_1omega', 'sin_1raan', 'cos_1raan', 'sin_1m', 'cos_1m']
for c in CORE_COLS:
    if c not in feature_names:
        raise ValueError(f"Critical column '{c}' not found in lifted_orbit_data.csv!")

target_cols = ['a', 'e', 'i', 'omega', 'raan', 'm']

# ==============================================================================
# 2. DYNAMIC TIME WINDOWS (3-FOLD EXPANDING CROSS VALIDATION)
# ==============================================================================
p60 = int(n_samples * 0.6)
p70 = int(n_samples * 0.7)
p80 = int(n_samples * 0.8)
p90 = int(n_samples * 0.9)

cv_splits = [
    {'train_end': p60, 'test_end': p70},
    {'train_end': p70, 'test_end': p80},
    {'train_end': p80, 'test_end': p90}
]

print("Dynamic CV Windows Generated:")
for i, s in enumerate(cv_splits):
    print(f"  Window {i+1}: Train [0:{s['train_end']}] -> Test [{s['train_end']}:{s['test_end']}]")

# ==============================================================================
# 3. REAL PHYSICAL SPACE (UNLIFTED) EVALUATION ENGINE
# ==============================================================================
def angle_error(true_vals, pred_vals):
    """Mean Absolute Error (MAE) for angles in radians."""
    return np.mean(np.abs(np.arctan2(np.sin(true_vals - pred_vals), np.cos(true_vals - pred_vals))))

def run_pipeline(Z_data, current_feature_names):
    """Trains the model and evaluates errors in the real unlifted space."""
    
    # Locate indices for inverse transformation
    idx = {col: current_feature_names.index(col) for col in CORE_COLS}
    
    feature_cv_errors = {col: [] for col in target_cols}
    max_eig_overall = 0
    is_stable_overall = True
    
    for split in cv_splits:
        Z_train = Z_data[:split['train_end']]
        
        # True future values
        df_raw_future = df_raw.iloc[split['train_end']:split['test_end']].reset_index(drop=True)
        N_forecast_steps = len(df_raw_future)
        
        Z1, Z2 = Z_train[:-1], Z_train[1:]
        z_current = Z_train[-1].copy()
        
        # Train Ridge EDMD
        model = Ridge(alpha=1e-7, fit_intercept=False)
        model.fit(Z1, Z2)
        K = model.coef_
        
        # Check Stability
        max_eig = np.max(np.abs(np.linalg.eigvals(K)))
        max_eig_overall = max(max_eig_overall, max_eig)
        if max_eig > 1.01:
            return {}, max_eig, False
            
        # Recursive Forecast
        Z_forecast = []
        for _ in range(N_forecast_steps):
            z_next = z_current @ K.T
            Z_forecast.append(z_next)
            z_current = z_next.copy()
        Z_forecast = np.array(Z_forecast)
        
        # Unlift Forecast
        pred_df = pd.DataFrame()
        pred_df[['a', 'e']] = scaler_ae.inverse_transform(Z_forecast[:, [idx['a'], idx['e']]])
        pred_df['i'] = Z_forecast[:, idx['i']]
        pred_df['omega'] = np.arctan2(Z_forecast[:, idx['sin_1omega']], Z_forecast[:, idx['cos_1omega']])
        pred_df['raan']  = np.arctan2(Z_forecast[:, idx['sin_1raan']],  Z_forecast[:, idx['cos_1raan']])
        pred_df['m']     = np.arctan2(Z_forecast[:, idx['sin_1m']],     Z_forecast[:, idx['cos_1m']])
        
        # Calculate MAE
        feature_cv_errors['a'].append(np.mean(np.abs(df_raw_future['a'].values - pred_df['a'].values)))
        feature_cv_errors['e'].append(np.mean(np.abs(df_raw_future['e'].values - pred_df['e'].values)))
        feature_cv_errors['i'].append(np.mean(np.abs(df_raw_future['i'].values - pred_df['i'].values)))
        feature_cv_errors['omega'].append(angle_error(df_raw_future['omega'].values, pred_df['omega'].values))
        feature_cv_errors['raan'].append(angle_error(df_raw_future['raan'].values, pred_df['raan'].values))
        feature_cv_errors['m'].append(angle_error(df_raw_future['m'].values, pred_df['m'].values))
            
    # Average across all CV windows
    mean_feature_errors = {col: np.mean(feature_cv_errors[col]) for col in target_cols}
    return mean_feature_errors, max_eig_overall, True

# ==============================================================================
# 4. BASELINE MODEL CALCULATION
# ==============================================================================
print("\n[Step 1] Calculating Baseline Model...")
base_feat_errs, base_eig, base_stable = run_pipeline(Z, feature_names)

print(f"Baseline Max Eigenvalue: {base_eig:.4f}")
print("Baseline Errors (Physical Dimensions):")
for k, v in base_feat_errs.items():
    print(f"  {k}: {v:.6e}")
print()

# ==============================================================================
# 5. AUTONOMOUS INTERACTION (ADDITION) LOOP
# ==============================================================================
combos = list(combinations(range(n_features), 2))
total_combos = len(combos)
print(f"[Step 2] Testing {total_combos} interaction combinations. This will take a few minutes...\n")

results = []
start_time = time.time()

for count, (idx1, idx2) in enumerate(combos):
    # Create the new interaction feature
    new_col = Z[:, idx1] * Z[:, idx2]
    Z_expanded = np.column_stack((Z, new_col))
    
    # Name mapping stays the same, we just conceptually appended to the end
    feat_errs, max_eig, stable = run_pipeline(Z_expanded, feature_names)
    
    if stable and feat_errs:
        total_pct_improvement = 0
        row_data = {
            'Feature Combination': f"({feature_names[idx1]} * {feature_names[idx2]})",
            'Feat1': feature_names[idx1],
            'Feat2': feature_names[idx2],
            'Max Eigenvalue': max_eig
        }
        
        # Calculate % improvement for each physical target (Negative = Improved)
        for target in target_cols:
            b_err = base_feat_errs[target]
            t_err = feat_errs[target]
            
            pct_change = ((t_err - b_err) / (b_err + 1e-12)) * 100
            row_data[f'{target} Change (%)'] = pct_change
            total_pct_improvement += pct_change
            
        overall_improvement = total_pct_improvement / len(target_cols)
        row_data['Overall Impact (%)'] = overall_improvement
        
        # Only keep features that improve the overall model by at least 0.5%
        if overall_improvement < -0.5:
            results.append(row_data)
            
    if (count + 1) % 500 == 0:
        elapsed = time.time() - start_time
        print(f"Processed: {count + 1}/{total_combos} | Good Terms Found: {len(results)} | Elapsed: {elapsed:.1f}s")

# ==============================================================================
# 6. RESULTS TABLE & AUTO-GENERATED CODE
# ==============================================================================
df_results = pd.DataFrame(results)

if not df_results.empty:
    df_results = df_results.sort_values(by='Overall Impact (%)', ascending=True).reset_index(drop=True)
    
    print("\n" + "="*80)
    print("✅ BEST INTERACTION TERMS (IMPROVES FORECAST IN REAL PHYSICS SPACE) ✅")
    print("Note: Negative (-) percentages indicate a REDUCTION in error (Green = Good).")
    print("="*80)
    
    def color_cells(val):
        if pd.isna(val) or not isinstance(val, (int, float)):
            return ''
        if val < -0.5:
            return 'color: #155724; background-color: #d4edda' # Green
        elif val > 0.5:
            return 'color: #721c24; background-color: #f8d7da' # Red
        return 'color: black'
        
    format_cols = [c for c in df_results.columns if '(%)' in c]
    format_dict = {col: "{:+.2f}%" for col in format_cols}
    format_dict['Max Eigenvalue'] = "{:.4f}"
    
    # Hide helper columns for clean display
    display_cols = [c for c in df_results.columns if c not in ['Feat1', 'Feat2']]
    display(df_results.head(40)[display_cols].style.map(color_cells, subset=format_cols).format(format_dict))

    print("\n" + "="*80)
    print("📋 AUTO-GENERATED CODE FOR LIFTING.IPYNB (TOP 15 ADDITIONS) 📋")
    print("Copy and paste these lines into the 'Feature Addition' section of your lifting code:\n")
    
    top_n = min(15, len(df_results))
    
    for i, row in df_results.head(top_n).iterrows():
        f1 = row['Feat1']
        f2 = row['Feat2']
        pct = row['Overall Impact (%)']
        
        print(f"# Adds an overall improvement of {abs(pct):.2f}%")
        print(f"df_final['({f1} * {f2})'] = df_final['{f1}'] * df_final['{f2}']")
        print()
    print("="*80)

else:
    print("\nNo feature combination was found that significantly improves the model.")

Loading data and initializing the Autonomous Pipeline...
Total Samples: 1862 | Base Features: 68
Dynamic CV Windows Generated:
  Window 1: Train [0:1117] -> Test [1117:1303]
  Window 2: Train [0:1303] -> Test [1303:1489]
  Window 3: Train [0:1489] -> Test [1489:1675]

[Step 1] Calculating Baseline Model...
Baseline Max Eigenvalue: 1.0003
Baseline Errors (Physical Dimensions):
  a: 3.988509e+01
  e: 1.360213e-05
  i: 6.533699e-06
  omega: 9.480808e-03
  raan: 6.257420e-06
  m: 1.566333e-02

[Step 2] Testing 2278 interaction combinations. This will take a few minutes...

Processed: 500/2278 | Good Terms Found: 12 | Elapsed: 79.0s
